In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_excel("dataset.xlsx", sheet_name="Dataset Utama")
print("Data siap:", df.shape)

Data siap: (1440, 63)


In [2]:
X = df.drop("disease", axis=1)   # X = semua gejala (input/petunjuk)
y = df["disease"]                # y = penyakit (jawaban yang ditebak)

print("Bentuk X:", X.shape)      # (1440, 62)
print("Bentuk y:", y.shape)      # (1440,)

Bentuk X: (1440, 62)
Bentuk y: (1440,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Data latih :", X_train.shape[0])
print("Data uji :", X_test.shape[0])

Data latih : 1152
Data uji : 288


In [4]:
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
print("Model sudah dilatih")

Model sudah dilatih


In [5]:
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average="weighted")

print("Akurasi :", round(acc, 3))
print("F1-Score:", round(f1,3))

Akurasi : 0.899
F1-Score: 0.896


In [6]:
import numpy as np

# Ambil 1 pasien dari data uji sebagai contoh
contoh = X_test.iloc[[0]]

# predict_proba = peluang model untuk SETIAP penyakit (bukan cuma 1)
proba = model.predict_proba(contoh)[0]
classes = model.classes_

# Urutkan dari peluang tertinggi, ambil 3 teratas
top3_idx = np.argsort(proba)[::-1][:3]

print("3 Tebakan Teratas:")
for i in top3_idx:
    print(f" - {classes[i]}: {round(proba[i]*100, 1)}%")

print("\nPenyakit asli pasien ini:", y_test.iloc[0])

3 Tebakan Teratas:
 - Tuberculosis: 75.0%
 - Type 2 Diabetes Mellitus: 10.0%
 - Asthma: 4.0%

Penyakit asli pasien ini: Tuberculosis


In [7]:
def prediksi_dengan_confidence(model, data_pasien, threshold=0.40):
    proba = model.predict_proba(data_pasien)[0]
    classes = model.classes_
    top3_idx = np.argsort(proba)[::-1][:3]

    confidence_tertinggi = proba[top3_idx[0]]
    top3 = [(classes[i], round(proba[i]*100, 1)) for i in top3_idx]

    if confidence_tertinggi < threshold:
        catatan = "⚠️ Keyakinan rendah — gejala mungkin terlalu umum/samar. Disarankan konsultasi langsung."
    else:
        catatan = "✅ Keyakinan cukup."

    return top3, catatan

# Uji fungsinya
top3, catatan = prediksi_dengan_confidence(model, X_test.iloc[[0]])
print("Top-3:", top3)
print(catatan)

Top-3: [('Tuberculosis', np.float64(75.0)), ('Type 2 Diabetes Mellitus', np.float64(10.0)), ('Asthma', np.float64(4.0))]
✅ Keyakinan cukup.


In [8]:
import joblib

# Simpan model
joblib.dump(model, "model_medagent.pkl")

# PENTING: simpan juga urutan kolom gejala
feature_names = list(X.columns)
joblib.dump(feature_names, "fitur_gejala.pkl")

print("Model & daftar fitur tersimpan! ✅")
print("Jumlah fitur:", len(feature_names))

Model & daftar fitur tersimpan! ✅
Jumlah fitur: 62


In [9]:
print(classification_report(y_test, pred))

                          precision    recall  f1-score   support

          Acute Diarrhea       0.60      0.50      0.55        12
                  Anemia       0.90      0.75      0.82        12
                  Asthma       0.86      1.00      0.92        12
                COVID-19       0.92      1.00      0.96        12
              Chickenpox       0.82      0.75      0.78        12
             Chikungunya       1.00      0.83      0.91        12
          Conjunctivitis       1.00      1.00      1.00        12
Dengue Hemorrhagic Fever       0.86      1.00      0.92        12
                    GERD       1.00      1.00      1.00        12
         Gastroenteritis       0.55      0.50      0.52        12
           Heart Failure       1.00      0.83      0.91        12
             Hepatitis A       0.89      0.67      0.76        12
             Hepatitis B       0.73      0.92      0.81        12
            Hypertension       0.85      0.92      0.88        12
         

In [10]:
model_loaded = joblib.load("model_medagent.pkl")
fitur_loaded = joblib.load("fitur_gejala.pkl")

# Coba prediksi lagi pakai model yang dimuat dari file
top3, catatan = prediksi_dengan_confidence(model_loaded, X_test.iloc[[0]])
print("Top-3 dari model tersimpan:", top3)
print("✅ Model berhasil dimuat & dipakai!")

Top-3 dari model tersimpan: [('Tuberculosis', np.float64(75.0)), ('Type 2 Diabetes Mellitus', np.float64(10.0)), ('Asthma', np.float64(4.0))]
✅ Model berhasil dimuat & dipakai!
